In [1]:
!pip install numpy pandas plotly dash jupyter_plotly_dash

In [3]:
import pandas as pd
import numpy as np
import dash
from plotly.subplots import make_subplots
from plotly import graph_objects as go
from jupyter_plotly_dash import JupyterDash
from dash import dcc, html, Input, Output

#jupyter_dash.inline_exceptions = False
#jupyter_dash.infer_jupyter_proxy_config()


In [ ]:
df = pd.read_csv("/BWI_Passenger_Data_-_January_2010_to_July_2013_20260321.csv")

#Clean up and white space in columne strings
df.columns = df.columns.str.strip()

# Create subplots
fig = make_subplots(rows=1, cols=3)

#Remove commas from numbers, and cast all numbers to int64
df = df.replace(',','', regex=True)
df.fillna(0, inplace=True)

for i in df.columns[2:]:
    df[i] = df[i].astype('int64')

# This block consolidates sub companies,
# like United - Express into United, for example
companies = [ "American", "Delta", "Southwest", "United", "US Airways" ]

for c in companies:
    cols = [ col for col in df.columns if col.lower().startswith(c.lower()) and
col.lower() != c.lower() ]
    for col in cols:
        df[c] = df[c] + df[col]
        df = df.drop(col, axis=1)

df_temp = df.drop(columns=["Year", "Month"], axis=1)

# Clean up some small airlines, to go from 50+ airlines to 6
columns_to_drop = df_temp.columns[df_temp.sum(numeric_only=True) < 400000]
df = df.drop(columns=columns_to_drop)

df_year = df.groupby(by="Year").sum(numeric_only=True)
df_year = df_year.reset_index()

In [ ]:
# Normaize data using min/max
# Southwest has a hub at BWI, so the passenger values
for col in df.columns[2:]:
    df[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

In [ ]:
# Set up for plots
df_date = df.copy()
date = [ m + " " +  str(y) for y, m in zip(df_date["Year"], df_date["Month"]) ]
df_date["date"] = date
#df_date = df_date.melt(id_vars="date", value_vars=df.columns[2:])

colors = { 'American': 'darkblue', 'Southwest': 'orange', 'Delta': 'red',
'US Airways': 'blue', 'United': 'slategrey', 'Jetblue': 'royalblue' }

app = dash.Dash(__name__)
app.layout = html.Div([
    html.H1("BWI Airport Passengers on Airlines per month")])


In [ ]:
  # Plot bar graph

html.H3("Bar graph of arline passengers per month")
for airline in df.columns[2:]:
    df_airline = df_date[["date", airline]]
    fig.add_traces(go.Bar(x=df_date['date'], y=df_date[airline],
    name=airline, marker_color=colors[airline]),
    rows=1, cols=1 )

In [ ]:
#Code for scatter plot
html.H3("Bar graph of arline passengers per month")
for airline in df.columns[2:]:
    df_airline = df_date[["date", airline]]
    fig.add_traces(
        go.Scatter(
        x=df_date["date"],
        y=df_date[airline],
        marker_color=colors[airline],
        name=airline,
        mode='lines'),
        rows=1, cols=2
)

In [ ]:

airlines = df.columns[2:]
for line in airlines:
    fig.add_trace(go.Scatter(
        y=df_date[line],
        x=df_date["date"],
        name=line,
        mode='markers',
        marker=dict(
          color=colors[line],
          sizemin=1,
        ),
        opacity=1),
        row=1, col=3 )

In [ ]:
fig.show()

In [ ]:
if __name__ == '__main__':
    app.run(jupyter_mode='jupyterlab')

Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
